<a href="https://colab.research.google.com/github/JamesMartinOU/PublicRedditSentimentAnalysis/blob/main/WriteYFinanceFactTableToSnowflake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Python libraries
!pip install mysql-connector-python yfinance pandas sqlalchemy snowflake-connector-python snowflake.sqlalchemy

In [6]:
# Import Python libraries
import mysql.connector
import pandas as pd
from sqlalchemy import create_engine
import snowflake.connector
from snowflake.sqlalchemy import URL
import yfinance as yf

In [ ]:
# --- MySQL Connection ---
conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

# --- Get Stock Symbols and Company Names from Reddit Posts ---
query = """
SELECT DISTINCT keyword AS stock_symbol, company_name
FROM reddit_posts
WHERE LENGTH(keyword) <= 5 AND keyword REGEXP '^[A-Z]+$'
"""
symbols_df = pd.read_sql(query, conn)
conn.close()

# Preview pulled symbols
print("🔍 Pulled symbols:")
print(symbols_df.head())

stock_symbols = symbols_df['stock_symbol'].tolist()

# --- Pull Weekly Avg Closing Prices from yfinance ---
start_date = "2011-01-01"
end_date = "2025-04-04"  # End of week Friday

# Dictionary to store weekly closing prices
weekly_avg_closing = {}

for symbol in stock_symbols:
    try:
        stock = yf.Ticker(symbol)
        hist = stock.history(start=start_date, end=end_date)

        if not hist.empty:
            # Resample weekly ending on Friday
            weekly_avg = hist['Close'].resample('W-FRI').mean()
            weekly_avg_closing[symbol] = weekly_avg
            print(f"✅ Pulled weekly prices for {symbol}")
        else:
            print(f"⚠️ No data for {symbol}")
    except Exception as e:
        print(f"❌ Failed for {symbol}: {e}")

# --- Combine into Wide DataFrame ---
df_weekly = pd.DataFrame(weekly_avg_closing)
df_weekly.reset_index(inplace=True)
df_weekly.rename(columns={"Date": "week_ending"}, inplace=True)

# --- Convert to Long Format ---
df_long = df_weekly.melt(
    id_vars='week_ending',
    var_name='stock_symbol',
    value_name='avg_close'
)

# Drop rows with missing price
df_long.dropna(subset=['avg_close'], inplace=True)

# --- Merge with company_name from MySQL ---
df_long = df_long.merge(symbols_df, on='stock_symbol', how='left')

# Preview final output
print("\n✅ Final merged data:")
print(df_long.head())

In [18]:


with sf_engine.connect() as conn:
    df_long.head(0).to_sql('weekly_closing_prices', con=conn, index=False, if_exists='replace')
    df_long.to_sql('weekly_closing_prices', con=conn, index=False, if_exists='append', method='multi')